# 📥 DeepFake-AI: Direct Cloud-to-Drive Kaggle Dataset Downloader
**Authors**: Dar Rehan Rasool (*Research Contributor*), Dr. Riyaz (*Supervisor*), Dr. Javad (*Supervisor*)  
**Affiliation**: Islamic University of Science and Technology (IUST), Kashmir  
**Target Repository**: [https://github.com/darrehanrasool/Deepfake.git](https://github.com/darrehanrasool/Deepfake.git)  

---

### 🎯 Purpose
This notebook downloads the **500GB Kaggle Deepfake Detection Challenge (DFDC)** dataset directly into your **Google Drive** using Google Colab Pro's high-speed cloud network (1-2 Gbps).

> ⚠️ **Zero Local Disk Usage**: The dataset is transferred directly from Kaggle's cloud servers into your Google Drive storage. **0 bytes** are downloaded to your local MacBook.
> 
> 🔄 **Resumable Chunk-by-Chunk**: DFDC is split into 50 chunks (`part_0.zip` to `part_49.zip`, ~9.5GB each). This notebook downloads chunk-by-chunk and skips already downloaded parts so you can safely stop and resume anytime without losing progress.

## Step 1: Install Kaggle CLI

In [ ]:
!pip install --upgrade -q kaggle
print("✓ Kaggle CLI installed successfully!")

## Step 2: Mount Your Google Drive
Run this cell and follow the authorization prompt to connect your Google Drive.

In [ ]:
from google.colab import drive
import os
import shutil
from pathlib import Path

drive.mount('/content/drive')

# Create target storage directory in Google Drive
DRIVE_TARGET_DIR = Path("/content/drive/MyDrive/DeepFake_Datasets/DFDC")
DRIVE_TARGET_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Google Drive mounted! Target dataset directory: {DRIVE_TARGET_DIR}")

# Check storage space available in Drive
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"📊 Google Drive Total Space:    {total / (1024**3):.2f} GB")
print(f"📊 Google Drive Used Space:     {used / (1024**3):.2f} GB")
print(f"📊 Google Drive Free Space:     {free / (1024**3):.2f} GB")

## Step 3: Configure Kaggle API Credentials

### How to get your `kaggle.json` token:
1. Go to [kaggle.com](https://www.kaggle.com) and log in.
2. Click your profile picture (top-right) -> **Settings**.
3. Scroll to the **API** section and click **Create New Token**.
4. This downloads a file named `kaggle.json`.
5. Place that `kaggle.json` file in your **Google Drive root** (`MyDrive/kaggle.json`), **OR** enter your credentials below.

In [ ]:
import json
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_dest = kaggle_dir / "kaggle.json"

# Method A: Auto-detect from Google Drive root
drive_kaggle = Path("/content/drive/MyDrive/kaggle.json")

if drive_kaggle.exists():
    shutil.copy(drive_kaggle, kaggle_dest)
    os.chmod(kaggle_dest, 0o600)
    print(f"✓ Found and configured Kaggle token from Google Drive: {drive_kaggle}")
else:
    print("kaggle.json not found in Google Drive root. Please enter credentials below:")
    KAGGLE_USERNAME = input("Enter Kaggle Username: ").strip()
    KAGGLE_KEY = input("Enter Kaggle API Key: ").strip()
    if KAGGLE_USERNAME and KAGGLE_KEY:
        with open(kaggle_dest, "w") as f:
            json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
        os.chmod(kaggle_dest, 0o600)
        # Also backup to Drive for future sessions
        shutil.copy(kaggle_dest, drive_kaggle)
        print(f"✓ Kaggle credentials saved and backed up to {drive_kaggle}!")

# Verify Kaggle API access and competition rules acceptance
!kaggle competitions list -s deepfake

## Step 4: Choose Chunks to Download
- The competition has 50 chunks (`part_0` to `part_49`), each ~9.5GB - 10GB.
- For a rapid start (~50GB), download parts `0-4`.
- For the full competition training set (~472GB), choose `all`.
- You can specify any range, e.g., `0-9` (100GB) or `10-19` (100GB).

In [ ]:
# Configuration parameters
COMPETITION_NAME = "deepfake-detection-challenge"

# Options:
# "0-4"    -> First 5 parts (~50 GB)
# "0-9"    -> First 10 parts (~100 GB)
# "all"    -> All 50 parts (~472 GB)
# "0,1,5"  -> Specific comma-separated parts
PARTS_TO_DOWNLOAD = "0-4"  # @param {type:"string"}

print(f"Selected parts to download: {PARTS_TO_DOWNLOAD}")
print(f"Target directory in Drive:  {DRIVE_TARGET_DIR}")

## Step 5: Execute Resumable Download Pipeline
This cell downloads directly to Google Drive. If a file is already downloaded, it automatically skips it.

In [ ]:
import subprocess
import time

# Parse parts
files_to_download = ["sample_submission.csv"]

if PARTS_TO_DOWNLOAD.lower() == "all":
    files_to_download.extend([f"dfdc_train_part_{i}.zip" for i in range(50)])
elif "-" in PARTS_TO_DOWNLOAD:
    start, end = map(int, PARTS_TO_DOWNLOAD.split("-"))
    for i in range(start, end + 1):
        files_to_download.append(f"dfdc_train_part_{i}.zip")
else:
    for p in PARTS_TO_DOWNLOAD.split(","):
        files_to_download.append(f"dfdc_train_part_{int(p.strip())}.zip")

print(f"Starting download of {len(files_to_download)} files directly to Google Drive...")

for idx, fn in enumerate(files_to_download, 1):
    target_file = DRIVE_TARGET_DIR / fn
    if target_file.exists() and target_file.stat().st_size > 1024 * 1024:
        size_gb = target_file.stat().st_size / (1024**3)
        print(f"[{idx}/{len(files_to_download)}] ⏩ Skipping {fn}: already in Drive ({size_gb:.2f} GB).")
        continue
        
    print(f"\n[{idx}/{len(files_to_download)}] ⬇️ Downloading {fn} directly to Google Drive...")
    t0 = time.time()
    cmd = [
        "kaggle", "competitions", "download",
        "-c", COMPETITION_NAME,
        "-f", fn,
        "-p", str(DRIVE_TARGET_DIR)
    ]
    res = subprocess.run(cmd)
    dt = time.time() - t0
    
    if target_file.exists():
        sz_mb = target_file.stat().st_size / (1024**2)
        print(f"✓ Completed {fn} ({sz_mb:.1f} MB in {dt:.1f}s, ~{sz_mb/max(dt,0.1):.1f} MB/s)")
    else:
        print(f"⚠️ Download attempt finished for {fn}. Verify destination folder.")

print("\n🎉 Download process completed!")

## Step 6: Verify Downloaded Dataset in Google Drive

In [ ]:
print("=" * 70)
print("📋 GOOGLE DRIVE DOWNLOAD AUDIT REPORT")
print("=" * 70)

downloaded_files = list(DRIVE_TARGET_DIR.glob("*"))
total_bytes = 0

for f in sorted(downloaded_files):
    sz = f.stat().st_size
    total_bytes += sz
    print(f"  • {f.name:32s} : {sz / (1024**3):6.2f} GB")

print("-" * 70)
print(f"Total Files Downloaded: {len(downloaded_files)}")
print(f"Total Data in Drive:    {total_bytes / (1024**3):.2f} GB")
print(f"Target Storage Path:    {DRIVE_TARGET_DIR}")
print("=" * 70)
print("✓ Ready for training on University GPU cluster next time!")